### Experimental vs simulated P(s)

Compares contact-probability-versus-genomic-separation curves between scsHi-C
and simulation data, for both *cis* (within a
chromatid) and *trans* (between the two sister chromatids).

Simulated inputs are the `cis_matrix.h5` / `trans_matrix.h5` files written by
`contact_map.py`.

In [ ]:
import cooler
import cooltools
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from polykit.analysis import contact_maps

#### Configuration

In [ ]:
# --- experimental data (Mitter et al. 2020, GSE152373) ---
EXP_ROOT = "../experimental_data"
EXP_CIS_COOL = f"{EXP_ROOT}/GSM4613674_G2.fc_1_2_3_4.wOldG2.cis.1000.cool"
EXP_TRANS_COOL = f"{EXP_ROOT}/GSM4613674_G2.fc_1_2_3_4.wOldG2.trans.1000.cool"

# --- simulated data ---
# Mirrors trajectory_dir() in contact_map.py, which is where the .h5 maps land.
SIM_ROOT = "../polymer_sister_chromatids/output"

CONTACT_RADIUS = 2.3        # must match CONTACT_RADIUS in contact_map.py

# Bin size of the simulated maps. BIN_MONOMERS must match NUMBER_OF_MAPS in
# contact_map.py -- if the two disagree the simulated x-axis is silently
# rescaled by a constant factor.
BIN_MONOMERS = 8
KB_PER_MONOMER = 2.5
BP_PER_BIN = BIN_MONOMERS * KB_PER_MONOMER * 1000

# Both experiment and simulation are normalised to their own cis P(s) at this
# separation, so the curves can be overlaid on one scale.
NORM_DIST_BP = 4e4

NPROC = 16


def sim_dir(condition, alpha, tau):
    """Directory holding one parameter point's contact maps."""
    return f"{SIM_ROOT}/{condition}_alpha_{alpha}_tau_{tau}h_beta0_gamma0"


# Parameter points to overlay. Add a row to plot another alpha.
SIM_RUNS = [
    # (alpha, tau, condition, colour)
    (100, 11, "WT12h", "#24a645"),
]

#### Helpers

In [ ]:
def norm_factor(dist_bp, ps, norm_dist=NORM_DIST_BP):
    """P(s) at the separation closest to `norm_dist`."""
    return ps[np.argmin(abs(dist_bp - norm_dist))]


def load_exp_ps(cool_path, nproc=NPROC):
    """P(s) from an experimental cooler, averaged over whole chromosomes.

    Returns (dist_bp, ps). The coolers are unbalanced, so `clr_weight_name` is
    None and the raw-count expected is used.
    """
    clr = cooler.Cooler(cool_path)

    # one whole-chromosome region per chromosome
    view_df = pd.DataFrame(
        [{"chrom": chrom, "start": 0, "end": length, "name": chrom}
         for chrom, length in clr.chromsizes.items()])

    cvd = cooltools.expected_cis(
        clr=clr,
        view_df=view_df,
        smooth=True,
        clr_weight_name=None,
        aggregate_smoothed=True,
        nproc=nproc,
    )

    return cvd["dist_bp"], cvd["count.avg.smoothed.agg"]


def load_sim_ps(path_name, kind, cutoff=CONTACT_RADIUS, nproc=NPROC):
    """P(s) from a simulated contact map. `kind` is 'cis' or 'trans'.

    Note that `expected_cis` is used for the trans maps too: the two sister
    chromatids share one coordinate system, so averaging along the diagonals
    gives inter-sister contact probability versus genomic separation.
    `expected_trans` is not applicable -- it is for pairs of different
    chromosomes, which have no common coordinate.
    """
    with h5py.File(f"{path_name}/cutoff{cutoff}/{kind}_matrix.h5", "r") as f:
        matrix = f[f"{kind}_matrix"][:]

    clr = contact_maps.coolify(matrix, "single_conformation")
    view_df = cooltools.lib.make_cooler_view(clr)

    cvd = cooltools.expected_cis(
        clr=clr,
        view_df=view_df,
        smooth=True,
        aggregate_smoothed=True,
        nproc=nproc,
    )

    dist_bp = cvd["dist"] * BP_PER_BIN
    # .copy() so the masking below is not a chained assignment on a view
    ps = cvd["balanced.avg.smoothed.agg"].copy()
    ps.loc[dist_bp < 2] = np.nan

    return dist_bp, ps

#### Load experimental P(s)

In [ ]:
exp_cis_dist, exp_cis_ps = load_exp_ps(EXP_CIS_COOL)
exp_trans_dist, exp_trans_ps = load_exp_ps(EXP_TRANS_COOL)

exp_norm = norm_factor(exp_cis_dist, exp_cis_ps)
print(f"experimental cis norm factor at {NORM_DIST_BP:.0e} bp: {exp_norm:.6g}")

#### Load simulated P(s)

In [ ]:
sim_curves = []

for alpha, tau, condition, colour in SIM_RUNS:
    path_name = sim_dir(condition, alpha, tau)

    cis_dist, cis_ps = load_sim_ps(path_name, "cis")
    trans_dist, trans_ps = load_sim_ps(path_name, "trans")

    # trans is divided by the *cis* norm factor on purpose, so that the
    # cis/trans ratio is preserved on the plot
    norm = norm_factor(cis_dist, cis_ps)

    sim_curves.append({
        "alpha": alpha,
        "tau": tau,
        "condition": condition,
        "colour": colour,
        "cis_dist": cis_dist,
        "cis_ps": cis_ps,
        "trans_dist": trans_dist,
        "trans_ps": trans_ps,
        "norm": norm,
    })

    print(f"alpha={alpha}, tau={tau}: cis norm factor {norm:.6g}")

#### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for run in sim_curves:
    alpha = run["alpha"]
    label = rf"Simu cis, $\alpha$={alpha}"

    ax.loglog(run["cis_dist"], run["cis_ps"] / run["norm"],
              color=run["colour"], linewidth=2, label=label)
    ax.loglog(run["trans_dist"], run["trans_ps"] / run["norm"],
              color=run["colour"], linewidth=1.5, linestyle="--")

ax.loglog(exp_cis_dist, exp_cis_ps / exp_norm,
          color="#A04294", linewidth=1.5, label="Exp cis")
ax.loglog(exp_trans_dist, exp_trans_ps / exp_norm,
          color="#A04294", linewidth=1.5, linestyle="--", label="Exp trans")

ax.set_xlim(4e4, 1e8)
ax.set_ylim(6e-5, 0.15e1)
ax.set_aspect("equal")

ax.tick_params(axis="both", which="major", length=9, width=1.5)
ax.tick_params(axis="both", which="minor", length=4, width=0.75)

ax.set_xlabel("Genomic distance [bp]", fontsize=26)
ax.set_ylabel("Contact frequency", fontsize=26)
ax.tick_params(labelsize=24)
ax.set_title("WT", fontsize=20)
ax.grid(True)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=14)

plt.show()